# TV2 – Học máy truyền thống (SMS spam tiếng Việt)
Chạy lần lượt từng ô (Shift + Enter). Cần `TV2_hoc_may.zip`: kéo thả vào thanh thư mục bên trái của Colab.

## 1. Giải nén gói

In [ ]:
!unzip -o -q TV2_hoc_may.zip
%cd TV2_hoc_may
!ls

## 2. Cài thư viện (Colab thường có sẵn, ô này để chắc chắn)

In [ ]:
!pip install -q imbalanced-learn

## 3. Chạy toàn bộ thí nghiệm (khoảng 1 phút)
Dùng file gốc của nhóm trong `data/raw`. Muốn dùng file khác: đổi đường dẫn `--train`, `--test` (cột `message` hoặc `text`, và `label` 0/1 hoặc `y`).

In [ ]:
!python train_ml.py --train data/raw/train_goc.csv --test data/raw/test_goc.csv --regex_set vi

## 4. Xem bảng kết quả

In [ ]:
import pandas as pd, json
pd.set_option('display.width', 200)
print('--- 8 tổ hợp ---')
display(pd.read_csv('results/ket_qua_ml_VI.csv'))
print('--- Xử lý mất cân bằng ---')
display(pd.read_csv('results/so_sanh_mat_can_bang_VI.csv'))
print('--- Ghép Regex ---')
display(pd.read_csv('results/ghep_regex.csv'))
r = json.load(open('results/ket_qua_cuoi.json'))
print('--- Kết quả cuối (test) ---'); print(r['test_cuoi'])
print('--- Theo nhóm mẫu câu ---'); print(r['test_theo_nhom_mau_cau'])

## 5. Xem biểu đồ

In [ ]:
from IPython.display import Image, display
for f in ['phan_bo_lop','do_dai_tin','so_sanh_8_to_hop','confusion_matrix','pr_curve']:
    display(Image(f'results/figures/{f}.png', width=480))

## 6. Thử mô hình đã lưu
Pipeline nhận list chuỗi; cần `features_regex.py` cùng thư mục. Tin nên ẩn danh giống dữ liệu train (số thành `[NUMBER]`, tiền thành `[MONEY]`).

In [ ]:
import joblib, json
model = joblib.load('model/model.joblib')
nguong = json.load(open('model/config.json'))['nguong']

tin = ['Chuc mung ban da trung thuong [MONEY], bam vao link de nhan qua',
       'Ma OTP xac thuc GD la [NUMBER], hieu luc [TIME]']
for t, s in zip(tin, model.predict_proba(tin)[:, 1]):
    print(f"{'SPAM' if s >= nguong else 'HAM ':4s}  điểm={s:.2f}  | {t}")

## 7. Tải kết quả về máy

In [ ]:
!zip -r -q TV2_ketqua.zip results model
from google.colab import files
files.download('TV2_ketqua.zip')